# Checks

Numerical checks of statements in the paper that no figure or table shows directly:
the construction of the model (Methods), the identities and theorems of the
Supplementary Material, and numbers quoted in the text. Each check prints the computed
value next to the value in the paper, and assertions stop the notebook if a
qualitative claim fails. The later sections read `data/generated/`. Run with
`make checks`.

## Model construction (Methods)
Transition matrices are stochastic; payoffs are linear in the two rewards
(Materials and Methods); exchanging the agents' rates exchanges their payoffs; and, against a
competitor that never explores, an agent faces the single-agent problem with the
distant reward doubled and the payoff halved (Results, "Strategies and payoffs").

In [1]:
import numpy as np
from scipy.sparse import identity, kron
from scipy.sparse.linalg import splu

from competitive_exploration.single_agent import site_occupancy, state_sites, transition_matrix
from competitive_exploration.two_agent import reward_shares, two_agent_occupancy

for d in range(1, 8):
    for eps in [0, 0.3, 1]:
        assert np.allclose(transition_matrix(d, eps).sum(axis=1), 1)

d, eps1, eps2, gamma, R0, Rd = 5, 0.3, 0.6, 1 - 1 / 5000, 1.3, 3.7
P1 = transition_matrix(d, eps1)
sites = state_sites(d)
direct = splu((identity(P1.shape[0]) - gamma * P1).tocsc()).solve(
    R0 * (sites == 0) + Rd * (sites == d)
)[0]
alpha, beta = site_occupancy(d, eps1, gamma)
assert np.isclose(direct, R0 * alpha + Rd * beta, rtol=1e-10)

K = kron(P1, transition_matrix(d, eps2), format="csc")
shares = reward_shares(d)
direct = splu(identity(K.shape[0], format="csc") - gamma * K).solve(
    R0 * shares[:, 0] + Rd * shares[:, 1]
)[0]
S = two_agent_occupancy(d, eps1, eps2, gamma)
assert np.isclose(direct, R0 * S[0, 0] + Rd * S[0, 1], rtol=1e-10)
assert np.allclose(two_agent_occupancy(d, eps2, eps1, gamma), S[::-1], rtol=1e-10)

for eps in [0.1, 0.4, 0.9]:
    S = two_agent_occupancy(d, 0.0, eps, gamma)
    alpha, beta = site_occupancy(d, eps, gamma)
    assert np.isclose(R0 * S[1, 0] + Rd * S[1, 1], (R0 * alpha + 2 * Rd * beta) / 2, rtol=1e-10)
print("all passed")

all passed


## The fast signaling computation (Figure 6, Table S2)
`signaling_fast`, which computes the signaling sweep of Figure 6 and Table S2, never forms
the $m^4$ payoff arrays of `signaling.signaling_payoff_grid`. On grids small enough for
those arrays, its payoff components reproduce them, and its Nash and locally stable
summaries match `equilibria.signaling_summary`.

In [2]:
import itertools

import numpy as np

from competitive_exploration.equilibria import signaling_summary
from competitive_exploration.signaling import signaling_payoff_grid
from competitive_exploration.signaling_fast import (
    combine,
    signaling_components,
    signaling_summary_fast,
)


def same_summary(full, fast):
    # Equal counts and dilemma flag, and payoffs equal to within 1e-9 (or both NaN).
    payoffs_match = all(
        (np.isnan(x) and np.isnan(y)) or abs(x - y) <= 1e-9 * abs(x)
        for x, y in zip(full[:2], fast[:2])
    )
    return payoffs_match and tuple(full[2:]) == tuple(fast[2:])


largest_difference, cases, mismatches = 0.0, 0, []
for d, T, m in itertools.product((1, 2, 3, 5, 8), (1000, 5000, 100000), (9, 17)):
    eps_grid = np.union1d(np.linspace(0, 1, m), [1e-4, 1e-2])
    gamma = 1 - 1 / T
    components = signaling_components(d, eps_grid, gamma)
    U_0 = signaling_payoff_grid(d, eps_grid, gamma, R0=1.0, Rd=0.0)
    U_d = signaling_payoff_grid(d, eps_grid, gamma, R0=0.0, Rd=1.0)
    for R in (2.1, 2.3, 3.0, 5.5, 12.0, 20.0):
        C, A, B = combine(components, 1.0, R)
        # U[k, i, a, j, b] = C[k, i, j] + A[k, i, j, a] + B[k, i, j, b]
        U = C[:, :, None, :, None] + A.transpose(0, 1, 3, 2)[:, :, :, :, None] + B[:, :, None, :, :]
        U_full = np.stack([U_0[k] + R * U_d[k] for k in range(2)])
        largest_difference = max(largest_difference, float(np.abs(U - U_full).max() / np.abs(U_full).max()))
        cases += 1
        if not same_summary(signaling_summary(U_full[0], U_full[1]), signaling_summary_fast(C, A, B)):
            mismatches.append((d, T, m, R))
print(f"payoff components: largest relative difference {largest_difference:.1e}; "
      f"summaries: {cases} cases, {len(mismatches)} mismatches")
assert largest_difference < 1e-10 and not mismatches

payoff components: largest relative difference 6.2e-16; summaries: 180 cases, 0 mismatches


## Two-agent payoffs from single-agent occupancies (Sections S4 and S5)
The paper checks Theorem 4 against direct solutions of the two-agent chain at 60
random parameter sets ($d = 1$–12, $1-\gamma$ between $10^{-5}$ and $10^{-2}$,
$R = 2.1$–20, both rates uniform on $[0, 1]$), and Theorem 3 at nine combinations of
distance and discount factor. The anti-coordination payoff of Section S4,
$\Pi(\varepsilon, 0) = R_0/(1-\gamma) + R_d\hat\beta(\varepsilon)$, is checked too.

In [3]:
import numpy as np
from scipy.sparse import identity, kron
from scipy.sparse.linalg import splu

from competitive_exploration.single_agent import site_occupancy, state_sites, transition_matrix
from competitive_exploration.two_agent import two_agent_occupancy


def payoff(d, eps_own, eps_other, gamma, R):
    """Payoff to an agent with rate eps_own against a competitor with eps_other (R0 = 1)."""
    S = two_agent_occupancy(d, eps_own, eps_other, gamma)
    return S[0, 0] + R * S[0, 1]


def joint_time(d, eps1, eps2, gamma, condition):
    """Discounted number of steps at which both agents' sites satisfy condition."""
    K = kron(transition_matrix(d, eps1), transition_matrix(d, eps2), format="csc")
    x1, x2 = np.meshgrid(state_sites(d), state_sites(d), indexing="ij")
    indicator = (condition(x1) & condition(x2)).ravel().astype(float)
    return splu(identity(K.shape[0], format="csc") - gamma * K).solve(indicator)[0]


rng = np.random.default_rng(0)
errors = []
for _ in range(60):
    d = int(rng.integers(1, 13))
    gamma = 1 - 10 ** rng.uniform(-5, -2)
    R = rng.uniform(2.1, 20)
    eps_new, eps = rng.uniform(0, 1, 2)
    # G(eps_new | eps) - G(eps_new | 0), where G(e | eps) = pi(e, eps) - pi(0, eps).
    lhs = (payoff(d, eps_new, eps, gamma, R) - payoff(d, 0, eps, gamma, R)) - (
        payoff(d, eps_new, 0, gamma, R) - payoff(d, 0, 0, gamma, R)
    )
    rhs = -joint_time(d, eps_new, eps, gamma, lambda x: x != 0) / 2 - R * joint_time(
        d, eps_new, eps, gamma, lambda x: x == d
    ) / 2
    errors.append(abs(lhs - rhs) / abs(lhs))
print(f"Theorem 4 identity: max relative error {max(errors):.1e} (paper: below 1e-8)")
assert max(errors) < 1e-8

def away_time(d, eps, gamma):
    """L(eps): discounted number of steps a lone agent spends away from site 0."""
    P = transition_matrix(d, eps)
    away = (state_sites(d) != 0).astype(float)
    return splu((identity(P.shape[0]) - gamma * P).tocsc()).solve(away)[0]


eps_grid = np.linspace(0, 1, 1001)[1:]
for d in [2, 5, 10]:
    for T in [1000, 5000, 100000]:
        gamma = 1 - 1 / T
        occupancy = np.array([site_occupancy(d, eps, gamma) for eps in eps_grid])
        L = np.array([away_time(d, eps, gamma) for eps in eps_grid])
        # L / beta-hat grows without bound as eps -> 0; rates at which beta-hat is
        # below solver precision are left out of the infimum.
        resolved = occupancy[:, 1] > 1e-9 / (1 - gamma)
        R_star = np.min(L[resolved] / occupancy[resolved, 1])
        # Gain from exploring at the best rate against a competitor that stays.
        S = np.array([two_agent_occupancy(d, eps, 0.0, gamma)[0] for eps in eps_grid])
        stay = 1 / (2 * (1 - gamma))
        nash = [np.max(S[:, 0] + f * R_star / 2 * S[:, 1]) <= stay for f in (0.99, 1.01)]
        print(f"d = {d:2d}, gamma = 1 - 1/{T}: R* = {R_star:7.3f}; both stay is Nash at "
              f"0.99 R*/2: {nash[0]}, at 1.01 R*/2: {nash[1]}")
        assert nash == [True, False]

        # Anti-coordination collective payoff (Section S4).
        for k in [100, 500]:
            pi_explorer = payoff(d, eps_grid[k], 0, gamma, 3.0)
            pi_stayer = payoff(d, 0, eps_grid[k], gamma, 3.0)
            assert np.isclose(pi_explorer + pi_stayer, 1 / (1 - gamma) + 3.0 * occupancy[k, 1])

Theorem 4 identity: max relative error 2.2e-09 (paper: below 1e-8)


d =  2, gamma = 1 - 1/1000: R* =   1.093; both stay is Nash at 0.99 R*/2: True, at 1.01 R*/2: False


d =  2, gamma = 1 - 1/5000: R* =   1.041; both stay is Nash at 0.99 R*/2: True, at 1.01 R*/2: False


d =  2, gamma = 1 - 1/100000: R* =   1.009; both stay is Nash at 0.99 R*/2: True, at 1.01 R*/2: False


d =  5, gamma = 1 - 1/1000: R* =   2.435; both stay is Nash at 0.99 R*/2: True, at 1.01 R*/2: False


d =  5, gamma = 1 - 1/5000: R* =   1.828; both stay is Nash at 0.99 R*/2: True, at 1.01 R*/2: False


d =  5, gamma = 1 - 1/100000: R* =   1.361; both stay is Nash at 0.99 R*/2: True, at 1.01 R*/2: False


d = 10, gamma = 1 - 1/1000: R* =   7.382; both stay is Nash at 0.99 R*/2: True, at 1.01 R*/2: False


d = 10, gamma = 1 - 1/5000: R* =   4.548; both stay is Nash at 0.99 R*/2: True, at 1.01 R*/2: False


d = 10, gamma = 1 - 1/100000: R* =   2.679; both stay is Nash at 0.99 R*/2: True, at 1.01 R*/2: False


## Environmental turnover (Section S6)
With turnover at rate $\kappa$, payoffs equal the static payoffs at discount factor
$\gamma(1-\kappa)$, multiplied by $(1-\gamma(1-\kappa))/(1-\gamma)$.

In [4]:
import numpy as np

from competitive_exploration.single_agent import transition_matrix
from competitive_exploration.two_agent import reward_shares, two_agent_occupancy

d, eps1, eps2, R, gamma, kappa = 4, 0.3, 0.5, 2.1, 1 - 1 / 5000, 1e-3
K = np.kron(transition_matrix(d, eps1).toarray(), transition_matrix(d, eps2).toarray())
reward = reward_shares(d)[:, 0] + R * reward_shares(d)[:, 1]

# Each step, with probability kappa both agents return to the start state.
restart = np.zeros_like(K)
restart[:, 0] = 1
V = np.linalg.solve(np.eye(len(K)) - gamma * ((1 - kappa) * K + kappa * restart), reward)[0]

reduced = gamma * (1 - kappa)
S = two_agent_occupancy(d, eps1, eps2, reduced)
static = S[0, 0] + R * S[0, 1]
factor = (1 - reduced) / (1 - gamma)
print(f"turnover payoff {V:.6f}; factor x static payoff at gamma (1 - kappa) {factor * static:.6f}")
assert np.isclose(V, factor * static, rtol=1e-9)

turnover payoff 4293.155147; factor x static payoff at gamma (1 - kappa) 4293.155147


## Nash equilibria at $d = 4$ (Results, "Strategies and payoffs" and "Nash equilibria", Figure 3)
At $\gamma = 1 - 1/5000$ there are three Nash equilibria at $R = 2.1$ (both explore and
the anti-coordination pair) and the anti-coordination pairs are no longer Nash at
$R = 2.15$. Player 2's best response drops to zero between $\varepsilon_1 = 0.27$ and
$0.31$. Computed on the 1001-point grid of Figure 3.

In [5]:
import numpy as np

from competitive_exploration.equilibria import best_response, classify_points, find_nash
from competitive_exploration.paths import DATA_DIR

data = np.load(DATA_DIR / "two_agent_payoffs.npz")
eps_grid, R = data["eps_grid"], float(data["R"])

counts = {}
for R_new in [2.1, 2.15]:
    Z = data["Z_0"] + R_new / R * data["Z_d"]
    symmetric, anti_coordination, both_stay, other = classify_points(find_nash(Z)[0])
    counts[R_new] = (len(symmetric), len(anti_coordination), len(both_stay) + len(other))
    print(f"R = {R_new}: {counts[R_new][0]} both-explore, {counts[R_new][1]} "
          f"anti-coordination, and {counts[R_new][2]} other Nash equilibria")
assert counts[2.1] == (1, 2, 0) and counts[2.15][1:] == (0, 0)

Z = data["Z_0"] + data["Z_d"]
response = eps_grid[best_response(Z)]
drop = np.flatnonzero((response[:-1] > 0) & (response[1:] == 0))[0]
print(f"best response drops to 0 between eps1 = {eps_grid[drop]:.3f} and "
      f"{eps_grid[drop + 1]:.3f} (paper: between 0.27 and 0.31)")

R = 2.1: 1 both-explore, 2 anti-coordination, and 0 other Nash equilibria
R = 2.15: 1 both-explore, 0 anti-coordination, and 0 other Nash equilibria
best response drops to 0 between eps1 = 0.299 and 0.300 (paper: between 0.27 and 0.31)


## The strict dilemma at $d = 4$, $R = 2.1$ (Results, "Competition produces a social dilemma", Figure 4)
The both-explore Nash equilibrium attains 84% of the collective payoff of the best
locally stable point, and the dynamics reach it from 83% of starting pairs.

In [6]:
import numpy as np

from competitive_exploration.equilibria import BOTH_EXPLORE, classify_points, find_nash
from competitive_exploration.paths import DATA_DIR

data = np.load(DATA_DIR / "two_agent_local_br.npz")
Z, codes, endpoints = data["Z"][0], data["codes"][0], data["endpoints"][0]
W = Z + Z.T
symmetric = classify_points(find_nash(Z)[0])[0]
ratio = W[symmetric[0]] / W[endpoints].max()
print(f"d = {data['d_values'][0]}, R = {data['R_values'][0]}: both-explore Nash attains "
      f"{100 * ratio:.1f}% of the best locally stable payoff (paper: 84%), reached from "
      f"{100 * np.mean(codes == BOTH_EXPLORE):.1f}% of starting pairs (paper: 83%)")

d = 4, R = 2.1: both-explore Nash attains 84.2% of the best locally stable payoff (paper: 84%), reached from 83.0% of starting pairs (paper: 83%)


## Locally stable equilibria in four regimes (Results, "Locally stable equilibria", Figures 4 and S8)
Each regime of Figure 4 has four locally stable equilibria: both staying, both exploring,
and the two anti-coordination points. So does the two-site model of Figure S8, where the
anti-coordination points are socially optimal and the both-explore point is reached from
most starting pairs.

In [7]:
import numpy as np

from competitive_exploration.equilibria import (
    ANTI_COORDINATION,
    BOTH_EXPLORE,
    BOTH_STAY,
    find_locally_stable,
    outcome_type,
)
from competitive_exploration.paths import DATA_DIR

four = [BOTH_STAY, BOTH_EXPLORE, ANTI_COORDINATION, ANTI_COORDINATION]
lattice = np.load(DATA_DIR / "two_agent_local_br.npz")
for k, (d, R) in enumerate(zip(lattice["d_values"], lattice["R_values"])):
    points, mask = find_locally_stable(lattice["Z"][k])
    assert sorted(outcome_type(*p) for p in points) == sorted(four)
    assert np.array_equal(mask, lattice["endpoints"][k])  # the circles in Figure 4
    print(f"d = {d}, R = {R:g}: locally stable points {points}")

two_site = np.load(DATA_DIR / "two_site_local_br.npz")
points, mask = find_locally_stable(two_site["Z"])
optimal = [tuple(p) for p in np.argwhere(two_site["social_optimum"])]
assert sorted(outcome_type(*p) for p in points) == sorted(four)
assert all(outcome_type(*p) == ANTI_COORDINATION for p in optimal)
print(f"two-site model: locally stable points {points}, socially optimal {optimal}; both explore "
      f"reached from {100 * np.mean(two_site['codes'] == BOTH_EXPLORE):.0f}% of starting pairs "
      f"(lattice: {100 * np.mean(lattice['codes'][0] == BOTH_EXPLORE):.0f}%)")

d = 4, R = 2.1: locally stable points [(0, 0), (0, 62), (55, 55), (62, 0)]
d = 6, R = 4: locally stable points [(0, 0), (0, 102), (100, 100), (102, 0)]
d = 8, R = 3: locally stable points [(0, 0), (0, 127), (122, 122), (127, 0)]
d = 20, R = 3: locally stable points [(0, 0), (0, 177), (173, 173), (177, 0)]
two-site model: locally stable points [(0, 0), (0, 68), (58, 58), (68, 0)], socially optimal [(np.int64(0), np.int64(68)), (np.int64(68), np.int64(0))]; both explore reached from 84% of starting pairs (lattice: 83%)


## Strict and Pareto dilemmas (Results, "Competition produces a social dilemma" and "Strict dilemmas usually involve a conflict of interest"; Table 2, Table S1)
In every strict dilemma the worst Nash equilibrium has both agents staying or both
exploring, and strict Pareto dilemmas occur with each of those types. Whenever
anti-coordination is a Nash equilibrium it is the socially optimal locally stable point,
and no both-explore point that is not Nash is socially optimal (the two 0.0% cells of
Table 2). At $d = 1$ the worst Nash equilibrium of every strict dilemma falls short of the
best locally stable point by at most 0.022% (Methods). Strict Pareto dilemmas are about
2% or fewer of strict dilemmas at every discount factor of Table S1.

In [8]:
import numpy as np

from competitive_exploration.equilibria import BOTH_EXPLORE, BOTH_STAY
from competitive_exploration.paths import DATA_DIR
from competitive_exploration.plotting import OUTCOME_LABELS

sweep = np.load(DATA_DIR / "sweep_nash_social_dilemma_fine401.npz")
strict, worst_type = sweep["strict_dilemma"], sweep["worst_nash_type"]
types, counts = np.unique(worst_type[strict], return_counts=True)
print("worst Nash equilibrium of strict dilemmas:",
      ", ".join(f"{OUTCOME_LABELS[k]} {n}" for k, n in zip(types, counts)))
assert set(types) <= {BOTH_STAY, BOTH_EXPLORE}
for code in [BOTH_STAY, BOTH_EXPLORE]:
    n = (strict & sweep["pareto_dilemma"] & (worst_type == code)).sum()
    print(f"strict Pareto dilemmas whose worst Nash equilibrium is {OUTCOME_LABELS[code]}: {n}")
    assert n > 0

# Rows: Nash, not Nash. Columns: socially optimal, not socially optimal.
symmetric = sweep["symmetric_counts"].sum(axis=(0, 1, 2))
anti_coordination = sweep["anti_coordination_counts"].sum(axis=(0, 1, 2))
print(f"anti-coordination Nash but not socially optimal: {anti_coordination[0, 1]}; "
      f"both explore not Nash but socially optimal: {symmetric[1, 0]}")
assert anti_coordination[0, 1] == 0 and symmetric[1, 0] == 0

a = list(sweep["d_values"]).index(1)
at_d1 = strict[:, a]
shortfall = 1 - sweep["worst_nash_payoff"][:, a][at_d1] / sweep["best_local_payoff"][:, a][at_d1]
print(f"d = 1: {at_d1.sum()} strict dilemmas, largest shortfall {100 * shortfall.max():.4f}% "
      f"(paper: at most 0.022%)")
assert shortfall.max() <= 0.00022

kinds = np.load(DATA_DIR / "dilemma_kinds.npz")
for t, T in enumerate(kinds["T_values"]):
    if T < 1000:  # Table S1 shows the seven longest horizons
        continue
    share = 100 * kinds["strict_pareto"][t].sum() / kinds["strict"][t].sum()
    print(f"gamma = 1 - 1/{T}: second-kind Pareto {100 * kinds['second_pareto'][t].mean():.1f}%, "
          f"strict Pareto {100 * kinds['strict_pareto'][t].mean():.1f}%, "
          f"Pareto share of strict {share:.2f}%")
    assert share <= 2

worst Nash equilibrium of strict dilemmas: both stay 4274, both explore 929
strict Pareto dilemmas whose worst Nash equilibrium is both stay: 58
strict Pareto dilemmas whose worst Nash equilibrium is both explore: 23
anti-coordination Nash but not socially optimal: 0; both explore not Nash but socially optimal: 0
d = 1: 4 strict dilemmas, largest shortfall 0.0215% (paper: at most 0.022%)
gamma = 1 - 1/1000: second-kind Pareto 92.3%, strict Pareto 0.9%, Pareto share of strict 1.90%
gamma = 1 - 1/5000: second-kind Pareto 98.2%, strict Pareto 0.4%, Pareto share of strict 1.48%
gamma = 1 - 1/10000: second-kind Pareto 99.3%, strict Pareto 0.1%, Pareto share of strict 0.57%
gamma = 1 - 1/50000: second-kind Pareto 100.0%, strict Pareto 0.2%, Pareto share of strict 1.29%
gamma = 1 - 1/100000: second-kind Pareto 100.0%, strict Pareto 0.1%, Pareto share of strict 0.79%
gamma = 1 - 1/500000: second-kind Pareto 100.0%, strict Pareto 0.1%, Pareto share of strict 0.61%
gamma = 1 - 1/1000000: second-

## Signaling at $d = 4$, $R = 2.1$ (Results, "Signaling can remove suboptimal equilibria", Table 3)
On the 51-rate grid of Table 3 the only Nash equilibrium with signaling has each agent
play $(0.36, 0)$. Its collective payoff, 11362, is 21% above that of the both-explore
Nash equilibrium without signaling (9357), above the largest collective payoff of any
strategy pair without signaling (11151 for rates that vary continuously, by Section S4),
and 97.8% of the largest collective payoff with signaling, 11612, which is not attained
at a Nash equilibrium. Every locally stable point with signaling other than both staying
beats the baseline's both-explore point.

In [9]:
import numpy as np
from scipy.optimize import minimize_scalar

from competitive_exploration.equilibria import nash_4d
from competitive_exploration.paths import DATA_DIR
from competitive_exploration.signaling import signaling_payoff_grid
from competitive_exploration.single_agent import site_occupancy

d, R, gamma = 4, 2.1, 1 - 1 / 5000
table = np.load(DATA_DIR / "locally_stable_points.npz")
eps_grid = table["eps_grid"]
U1, U2 = signaling_payoff_grid(d, eps_grid, gamma, Rd=R)
W = U1 + U2

nash = np.argwhere(nash_4d(U1, U2))
print("Nash equilibria with signaling, ((eps1, eps1'), (eps2, eps2')):",
      [tuple(eps_grid[p].round(2)) for p in nash])
assert len(nash) == 1
with_signaling = W[tuple(nash[0])]
baseline_nash = table["baseline_payoff"][table["baseline_nash"]].min()
print(f"collective payoff {with_signaling:.0f} (paper: 11362); worst Nash without signaling "
      f"{baseline_nash:.0f} (paper: 9357); rise {100 * (with_signaling / baseline_nash - 1):.1f}% "
      f"(paper: 21%)")
print(f"largest collective payoff with signaling {W.max():.0f} (paper: 11612), of which the "
      f"Nash equilibrium attains {100 * with_signaling / W.max():.1f}% (paper: 97.8%)")

# Without signaling, by Section S4 the best pair is the anti-coordination pair that
# maximizes beta-hat; the best pair on Figure 3's fine grid agrees.
eps = minimize_scalar(lambda e: -site_occupancy(d, e, gamma)[1], bounds=(0.1, 0.6),
                      method="bounded").x
best_baseline = 1 / (1 - gamma) + R * site_occupancy(d, eps, gamma)[1]
fine = np.load(DATA_DIR / "two_agent_payoffs.npz")
Z = fine["Z_0"] + fine["Z_d"]
print(f"best collective payoff without signaling {best_baseline:.0f} at ({eps:.3f}, 0); "
      f"best on the grid of Figure 3 {(Z + Z.T).max():.0f}")
assert with_signaling > best_baseline

stable = table["signaling_locally_stable"] & table["signaling_eps"].any(axis=1)
print(f"lowest payoff among locally stable signaling points other than both staying: "
      f"{table['signaling_payoff'][stable].min():.0f}")
assert table["signaling_payoff"][stable].min() > baseline_nash

Nash equilibria with signaling, ((eps1, eps1'), (eps2, eps2')): [(np.float64(0.36), np.float64(0.0), np.float64(0.36), np.float64(0.0))]
collective payoff 11362 (paper: 11362); worst Nash without signaling 9357 (paper: 9357); rise 21.4% (paper: 21%)
largest collective payoff with signaling 11612 (paper: 11612), of which the Nash equilibrium attains 97.8% (paper: 97.8%)
best collective payoff without signaling 11151 at (0.330, 0); best on the grid of Figure 3 11151
lowest payoff among locally stable signaling points other than both staying: 9477


## Signaling across the sweep (Results, "Signaling can remove suboptimal equilibria", Figure 6, Table S2, Section S8.2)
Where signaling removes a dilemma, the collective payoff at the worst Nash equilibrium
rises by as much as 21%. Signaling removes 45.5–100% of Pareto dilemmas and 7.7–28.7%
of the others. The counts of Section S8.2 and of the Table S2 caption, the two removed
dilemmas that lie partly outside the bottom row of Figure 6, and the example at $d = 2$,
$\gamma = 1 - 1/100000$, $R = 18.56$, in which the payoff of the agent that stays falls
for deviations up to $10^{-6}$ and rises for deviations of $10^{-5}$ or more, are
checked too.

In [10]:
import numpy as np

from competitive_exploration.equilibria import find_locally_stable
from competitive_exploration.paths import DATA_DIR
from competitive_exploration.two_agent import payoff_grid

base = np.load(DATA_DIR / "sweep_nash_social_dilemma_fine401.npz")
signaling = np.load(DATA_DIR / "signaling_fine401.npz")
candidate, pareto = signaling["candidate"], base["pareto_dilemma"]
solved = candidate & (signaling["status"] == 0)
worst = signaling["worst_nash_payoff"]
improvement = 100 * (worst - base["worst_nash_payoff"]) / base["worst_nash_payoff"]
print(f"largest worst-Nash improvement {improvement[solved].max():.1f}% (paper: 21%)")
for label, group in [("Pareto", pareto), ("collective only", ~pareto)]:
    share = [100 * solved[t][candidate[t] & group[t]].mean() for t in range(3)]
    print(f"{label}: removed {min(share):.1f}-{max(share):.1f}%")

# Section S8.2 and the Table S2 caption.
remains = candidate & np.isfinite(worst)
shown = np.zeros_like(candidate)  # as in Figure 6
for t, a in np.ndindex(candidate.shape[:2]):
    shown[t, a, np.flatnonzero(remains[t, a])[::7]] = True
counts = {
    "candidates": candidate.sum(axis=(1, 2)),
    "with a Nash equilibrium after signaling": remains.sum(axis=(1, 2)),
    "without one": (candidate & ~remains).sum(axis=(1, 2)),
    "shown in Figure 6": shown.sum(axis=(1, 2)),
    "collective only": (candidate & ~pareto).sum(axis=(1, 2)),
    "collective only, removed": (solved & ~pareto).sum(axis=(1, 2)),
    "Pareto": (candidate & pareto).sum(axis=(1, 2)),
    "Pareto, removed": (solved & pareto).sum(axis=(1, 2)),
}
paper = {
    "candidates": [128, 252, 545],
    "with a Nash equilibrium after signaling": [123, 242, 531],
    "without one": [5, 10, 14],
    "shown in Figure 6": [24, 43, 83],
    "collective only": [122, 246, 534],
    "collective only, removed": [35, 57, 41],
    "Pareto": [6, 6, 11],
    "Pareto, removed": [5, 6, 5],
}
for key, value in counts.items():
    print(f"{key}: {value.tolist()} (paper: {paper[key]})")
    assert value.tolist() == paper[key]

# Removed dilemmas shown in Figure 6 that fall partly outside its fixed bottom-row limits.
LIMITS = {1000: (1834, 2191), 5000: (9211, 11445), 100000: (190217, 225800)}
for t, T in enumerate(base["T_values"]):
    lo, hi = LIMITS[int(T)]
    values = np.stack([base["best_local_payoff"][t], base["worst_nash_payoff"][t], worst[t]])
    outside = shown[t] & solved[t] & ((values < lo) | (values > hi)).any(axis=0)
    for a, r in np.argwhere(outside):
        print(f"gamma = 1 - 1/{T}: d = {base['d_values'][a]}, R = {base['R_values'][r]:.2f} lies "
              f"partly outside the plotted range (baseline worst Nash {values[1, a, r]:.0f})")

# The example of Section S8.2.
T, d = 100000, 2
r = int(np.argmin(np.abs(base["R_values"] - 18.56)))
eps_grid = base["eps_grid"]
B0, Bd = payoff_grid(d, eps_grid, 1 - 1 / T)
Z = B0 + base["R_values"][r] * Bd
explorer = max(i for i, j in find_locally_stable(Z)[0] if j == 0)  # the pair (eps, 0)
for deviation in [1e-8, 1e-7, 1e-6, 1e-5, 1e-4]:
    gain = Z[list(eps_grid).index(deviation), explorer] - Z[0, explorer]
    print(f"stayer deviating to {deviation:g} against eps = {eps_grid[explorer]:g}: gain {gain:+.3g}")
    assert (gain < 0) == (deviation <= 1e-6)

largest worst-Nash improvement 20.9% (paper: 21%)
Pareto: removed 45.5-100.0%
collective only: removed 7.7-28.7%
candidates: [128, 252, 545] (paper: [128, 252, 545])
with a Nash equilibrium after signaling: [123, 242, 531] (paper: [123, 242, 531])
without one: [5, 10, 14] (paper: [5, 10, 14])
shown in Figure 6: [24, 43, 83] (paper: [24, 43, 83])
collective only: [122, 246, 534] (paper: [122, 246, 534])
collective only, removed: [35, 57, 41] (paper: [35, 57, 41])
Pareto: [6, 6, 11] (paper: [6, 6, 11])
Pareto, removed: [5, 6, 5] (paper: [5, 6, 5])
gamma = 1 - 1/1000: d = 4, R = 2.28 lies partly outside the plotted range (baseline worst Nash 1792)
gamma = 1 - 1/100000: d = 3, R = 9.03 lies partly outside the plotted range (baseline worst Nash 878109)


stayer deviating to 1e-08 against eps = 0.0325: gain -0.000987
stayer deviating to 1e-07 against eps = 0.0325: gain -0.00947
stayer deviating to 1e-06 against eps = 0.0325: gain -0.0561
stayer deviating to 1e-05 against eps = 0.0325: gain +3.3
stayer deviating to 0.0001 against eps = 0.0325: gain +419


## Hitting statistics (Discussion, Figure S9)
At $d = 4$ the probability of reaching site $d$ before returning to site 0 stays below
0.01 until the exploration rate is about 0.4, and reaches $1/d = 0.25$ at
$\varepsilon = 1$.

In [11]:
import numpy as np

from competitive_exploration.paths import DATA_DIR

data = np.load(DATA_DIR / "hitting_stats.npz")
probability, eps_grid = data["exact"][:, 0], data["eps_grid"]
print(f"P(hit d first) < 0.01 for eps up to {eps_grid[probability < 0.01].max():.2f}; "
      f"at eps = 1 it is {probability[-1]:.3f}")

P(hit d first) < 0.01 for eps up to 0.38; at eps = 1 it is 0.250


## Threshold reward ratio (Section S2, Figure S2)
The expressions for the expected time to reach site $d$, $E[\tau]$, and the fraction of
time spent at site $d$ afterwards, $f_d$, agree with direct solves of the lone agent's
chain. $R^* = 1$ at $d = 1$ for every horizon, $R^*$ decreases as the horizon grows, and
$R^*/d$ increases with $d$ from $d = 4$ at $\gamma = 1 - 1/1000$ and from $d = 7$ at
$\gamma = 1 - 1/100000$ onward. The scaling form $R^* \approx 2d/\ln(T/d^2)$ is printed
next to the exact threshold.

In [12]:
import numpy as np

from competitive_exploration.paths import DATA_DIR
from competitive_exploration.single_agent import transition_matrix

for d in [2, 5, 12]:
    for eps in [0.4, 0.7, 0.95]:
        P = transition_matrix(d, eps).toarray()
        visits = np.linalg.inv(np.eye(d) - P[:d, :d])[0]  # before reaching d, from site 0
        values, vectors = np.linalg.eig(P[d + 1 :, d + 1 :].T)  # after reaching d
        stationary = np.real(vectors[:, np.argmin(np.abs(values - 1))])
        f_d = stationary[d] / stationary.sum()
        delta, rho = 1 - eps, eps / (2 - eps)
        assert np.isclose(visits.sum(), (rho**-d - 1 - 2 * d * delta) / (2 * delta**2), rtol=1e-8)
        assert np.isclose(f_d, delta / (1 - (1 - delta**2) * rho**d), rtol=1e-8)
print("E[tau] and f_d agree with the chain")

data = np.load(DATA_DIR / "threshold_sweep.npz")
R_star, d_values = data["R_star"], data["d_values"]
print(f"Figure S2: {len(d_values)} distances, {len(data['T_values'])} horizons, "
      f"{len(data['eps_grid'])} exploration rates")
assert np.allclose(R_star[:, 0], 1)
assert np.all(np.diff(R_star[:, 1:], axis=0) <= 0)
print("R* = 1 at d = 1, and R* decreases with the horizon at every d >= 2")

onset = {}
for T, exact in zip(data["T_paper"], data["R_star_paper"]):
    shrinking = np.flatnonzero(np.diff(exact / d_values) <= 0)
    onset[int(T)] = int(d_values[shrinking.max() + 1]) if len(shrinking) else 1
    scaling = {d: 2 * d / np.log(T / d**2) for d in [10, 20, 30]}
    print(f"gamma = 1 - 1/{T}: R*/d increases from d = {onset[int(T)]} onward; "
          + ", ".join(f"d = {d}: R* = {exact[d - 1]:.2f}, 2d/ln(T/d^2) = {s:.2f}"
                      for d, s in scaling.items()))
assert onset[1000] == 4 and onset[100000] == 7

E[tau] and f_d agree with the chain
Figure S2: 40 distances, 85 horizons, 410 exploration rates
R* = 1 at d = 1, and R* decreases with the horizon at every d >= 2
gamma = 1 - 1/1000: R*/d increases from d = 4 onward; d = 10: R* = 7.38, 2d/ln(T/d^2) = 8.69, d = 20: R* = 28.51, 2d/ln(T/d^2) = 43.65, d = 30: R* = 68.08, 2d/ln(T/d^2) = 569.47
gamma = 1 - 1/5000: R*/d increases from d = 5 onward; d = 10: R* = 4.55, 2d/ln(T/d^2) = 5.11, d = 20: R* = 14.02, 2d/ln(T/d^2) = 15.84, d = 30: R* = 29.47, 2d/ln(T/d^2) = 34.99
gamma = 1 - 1/100000: R*/d increases from d = 7 onward; d = 10: R* = 2.68, 2d/ln(T/d^2) = 2.90, d = 20: R* = 6.54, 2d/ln(T/d^2) = 7.24, d = 30: R* = 11.57, 2d/ln(T/d^2) = 12.74


## Figure captions
Figure S4: the difference panels span about $\pm 0.03$, and the region with no symmetric
locally stable point is absent at the longest horizon. Figure S5: the bottom row's scale
begins near 0.67. Figure S6: the dynamics alternate between two strategy pairs at 1052
of the 18000 parameter combinations, from at most 4.4% of starting pairs at any one of
them (0.1% overall), and excluding those trajectories changes the ratio by at most 0.005.

In [13]:
import numpy as np

from competitive_exploration.local_br import local_br_endpoints
from competitive_exploration.paths import DATA_DIR

sweep = np.load(DATA_DIR / "sweep_nash_social_dilemma_fine.npz")
single = np.load(DATA_DIR / "local_max_eps_single.npz")
print(f"S4: largest |difference| {np.nanmax(np.abs(sweep['symmetric_eps'] - single['local_max_eps'])):.3f}; "
      f"combinations without a symmetric point, by horizon: {np.isnan(sweep['symmetric_eps']).sum(axis=(1, 2))}")
assert not np.isnan(sweep["symmetric_eps"][-1]).any()

small_d = np.load(DATA_DIR / "sweep_nash_social_dilemma_small_d.npz")
ratio = sweep["worst_nash_payoff"] / sweep["global_optimum_payoff"]
ratio[:, np.isin(sweep["d_values"], small_d["d_values"])] = (
    small_d["worst_nash_payoff"] / small_d["global_optimum_payoff"]
)
print(f"S5: smallest worst Nash / global optimum {np.nanmin(ratio):.3f}")

outcomes = np.load(DATA_DIR / "local_br_outcomes.npz")
cycles = outcomes["cycle_fraction"]
baseline = np.load(DATA_DIR / "sweep_nash_social_dilemma.npz")
change = 0.0
for index in zip(*np.nonzero(cycles > 0)):
    Z = baseline["Z"][index]
    end_i, end_j, fixed = local_br_endpoints(Z)
    payoff = (Z + Z.T)[end_i, end_j]
    change = max(change, abs(payoff.mean() - payoff[fixed].mean()) / baseline["global_optimum_payoff"][index])
print(f"S6: cycles at {(cycles > 0).sum()} of {cycles.size} combinations, from at most "
      f"{100 * cycles.max():.1f}% of starting pairs ({100 * cycles.mean():.1f}% overall); "
      f"excluding them changes the ratio by at most {change:.4f}")

S4: largest |difference| 0.030; combinations without a symmetric point, by horizon: [418  87   0]
S5: smallest worst Nash / global optimum 0.672


S6: cycles at 1052 of 18000 combinations, from at most 4.4% of starting pairs (0.1% overall); excluding them changes the ratio by at most 0.0052
